# Ragas evaluation: conversation version v7

## Scope and method
This evaluation scores the saved `app_ollama_conversation_v7.py` equivalent graph run in `results/diagnostic_conversation_v3_20261126_225338_066400.json`: V4's 36-recipe corpus, `chef.conversation_v6`, `chef.safety_rails_v8`, sections chunking and top-k 5. Answers were generated by local Ollama `qwen2.5:7b`; the same model judged them using Ragas, temperature 0, 8,192 context and 1,024 predicted-token cap. The Ragas run was made on 26 November 2026. It uses the already saved run of the exact graph rather than regenerating answers; each of the 20 cases used a fresh conversation.

The original development set has 20 cases. Thirteen expect retrieval and all 13 had an answer and context and were scored on every metric. Four conversion cases, two scaling cases and one greeting are not scored by these three retrieval metrics. The source diagnostic reports expected routes 20/20 and exact expected tool results 6/6; these checks do not verify the factual quality of all answers. No multi-turn conversation was scored by Ragas. Prior live conversation tests are documented in `REALISTIC_CONVERSATIONS_v3.md`.

## Overall Ragas results
| Metric | Mean | Valid scores |
| --- | ---: | ---: |
| Context precision with reference | 0.832 | 13/13 |
| Context recall | 0.910 | 13/13 |
| Faithfulness | 0.706 | 13/13 |

No case or metric was missing. These are arithmetic means on a small development set, not proof of safety, answer correctness or general performance. The same small local model generated and judged answers, which can introduce correlated errors. Reference answers and expected section labels are source-derived and not independently reviewed. Output caps and single-run model variance may affect judging. The earlier 12-case Ragas run used a different answer set and excluded a blocked tortilla case, so its averages are not a controlled comparison.

## Case scores
| Case | Context precision | Context recall | Faithfulness |
| --- | ---: | ---: | ---: |
| Q01 | 0.500 | 1.000 | 0.000 |
| Q02 | 1.000 | 1.000 | 1.000 |
| Q03 | 0.478 | 1.000 | 1.000 |
| Q04 | 0.500 | 0.909 | 0.000 |
| Q05 | 0.583 | 0.917 | 0.083 |
| Q06 | 0.806 | 0.500 | 1.000 |
| Q07 | 1.000 | 1.000 | 1.000 |
| Q08 | 1.000 | 0.500 | 1.000 |
| Q09 | 1.000 | 1.000 | 1.000 |
| Q10 | 1.000 | 1.000 | 0.667 |
| Q11 | 1.000 | 1.000 | 1.000 |
| Q12 | 1.000 | 1.000 | 0.429 |
| Q13 | 0.950 | 1.000 | 1.000 |

## Failure investigation
- **Q01 Ratatouille, Q04 Dal tadka, Q05 Borscht:** the app formats the ingredient list directly from its stored structured recipe. Its five displayed/retrieved chunks did not contain the corresponding `ingredients` section for these three cases: Q01 and Q04 scored 0.000 faithfulness, and Q05 scored 0.083. The absence of that evidence is confirmed from the exact saved metadata; it is a traceability and Ragas-context problem. The source recipe data can contain the quantities even though the retrieved chunks do not show them. A low Ragas score is not by itself proof that the formatted quantities are wrong. The additional context audit found the labelled evidence section in 10/13 cases. [Answer screenshot](results/screenshots_ragas_conversation_v1/01_ratatouille_answer.jpg) and [first retrieved chunk](results/screenshots_ragas_conversation_v1/02_ratatouille_retrieval_trace.jpg) show the case; the screenshot is only one chunk, while the JSON audit covers all five.
- **Q12 Spanish tortilla:** faithfulness was 0.429. The final answer deliberately adds a 71°C egg-safety adaptation based on FoodSafety.gov, while the retrieved recipe context and source-derived reference contain the original loose-centre instruction and do not contain that guidance. This is a known mismatch between the evidence provided to the judge and the adapted answer. The metric does not establish that the safety adaptation is incorrect. The separate tortilla report documents its rationale and external source. A future evaluation should include appropriately sourced safety context and review a suitable reference.
- **Q10 Miso soup:** faithfulness was 0.667. The source has a known discrepancy between its description and structured timing fields, and the structured answer warns that descriptions or steps may differ. The score warrants review; no independent adjudication of the discrepancy was performed here.
- **Q06 and Q08:** recall was 0.500 in each, despite their source-tag chunks being present. The references also include a verification caveat. This is an example where a small LLM judge and reference wording require human review; no single cause was established.

## Evidence and reproducibility
- Answer run: `results/diagnostic_conversation_v3_20261126_225338_066400.json`.
- Raw Ragas per-case scores and judge inputs: `results/ragas_ollama_diagnostic_v2_20261126_225737_961800.json`.
- Machine-readable summary: `results/ragas_conversation_summary_v1_20261126_230037_317400.json`.
- Section-label context audit: `results/ragas_context_audit_v1_20261126_225910_809000.json`.
- Scoring script: `ragas_ollama_diagnostic_v2.py`; summary and context-audit scripts are embedded in `32_ragas_conversation_v1.ipynb`. Running the scorer again consumes local compute and creates a new file. Paths and model settings must be kept consistent for comparison.

The next practical improvement is to include the exact structured ingredient evidence that the app uses in the context and source trace, then rerun a controlled evaluation on an independently reviewed set. This report records the result as measured; it does not alter the evaluated app.


## Screenshots
![Ratatouille answer](results/screenshots_ragas_conversation_v1/01_ratatouille_answer.jpg)

![First retrieved chunk](results/screenshots_ragas_conversation_v1/02_ratatouille_retrieval_trace.jpg)

## ragas_ollama_diagnostic_v2.py
Reproducible scoring or audit script.

In [ ]:
%%writefile ragas_ollama_diagnostic_v2.py
"""Score saved local answers in resumable, separately saved Ragas batches."""
import argparse
import json
import math
import os
import time
from pathlib import Path
os.environ['LANGCHAIN_TRACING_V2']='false'
os.environ['LANGSMITH_TRACING']='false'
from langchain_ollama import ChatOllama
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, Faithfulness
from ragas.run_config import RunConfig

ROOT=Path(__file__).resolve().parent

def main():
    p=argparse.ArgumentParser(); p.add_argument('run',type=Path); p.add_argument('--offset',type=int,default=0); p.add_argument('--limit',type=int,default=1); args=p.parse_args()
    data=json.loads(args.run.read_text(encoding='utf-8'))
    cases=[c for c in data['cases'] if c.get('expected_route')=='retrieve' and c.get('actual_route')=='retrieve' and c.get('contexts') and c.get('answer')]
    selected=cases[args.offset:args.offset+args.limit]
    output={'source_run':str(args.run.resolve()),'judge':'qwen2.5:7b','note':'Local generator and judge use the same model. Blocked cases are excluded, not treated as passes. References are not independently verified.', 'cases':[]}
    path=ROOT/'results'/f"ragas_ollama_diagnostic_v2_{time.strftime('%Y%m%d_%H%M%S')}_{time.time_ns()%1000000:06d}.json"
    for case in selected:
        row={'id':case['id']}
        # Each evaluate call owns an event loop; a new client avoids reusing a closed loop.
        judge=LangchainLLMWrapper(ChatOllama(model=os.getenv('OLLAMA_MODEL','qwen2.5:7b'), base_url='http://127.0.0.1:11434', temperature=0,num_ctx=8192,num_predict=1024))
        try:
            sample=SingleTurnSample(user_input=case['question'],reference=case['reference'],response=case['answer'],retrieved_contexts=case['contexts'])
            scored=evaluate(EvaluationDataset(samples=[sample]), metrics=[LLMContextPrecisionWithReference(),LLMContextRecall(),Faithfulness()], llm=judge, run_config=RunConfig(max_workers=1,max_retries=1,timeout=180),raise_exceptions=False)
            row['scores']=json.loads(scored.to_pandas().to_json(orient='records'))[0]
        except Exception as exc: row['error']=f'{type(exc).__name__}: {exc}'
        output['cases'].append(row); path.write_text(json.dumps(output,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
        print(case['id'], 'scores', {k:v for k,v in row.get('scores',{}).items() if k in ('llm_context_precision_with_reference','context_recall','faithfulness')}, 'error',row.get('error'),flush=True)
    print('Saved:',path,flush=True)

if __name__=='__main__': main()


## summarize_ragas_conversation_v1.py
Reproducible scoring or audit script.

In [ ]:
%%writefile summarize_ragas_conversation_v1.py
"""Summarize Ragas without hiding missing scores or excluded cases."""
import argparse,json,statistics,time
from pathlib import Path
ROOT=Path(__file__).resolve().parent
METRICS=('llm_context_precision_with_reference','context_recall','faithfulness')
def main():
 p=argparse.ArgumentParser();p.add_argument('source',type=Path);p.add_argument('ragas',type=Path);a=p.parse_args()
 run=json.loads(a.source.read_text(encoding='utf-8'));scored=json.loads(a.ragas.read_text(encoding='utf-8'))
 expected=[c['id'] for c in run['cases'] if c.get('expected_route')=='retrieve']
 cases={c['id']:c for c in scored.get('cases',[])}
 result={'source_run':str(a.source.resolve()),'ragas_run':str(a.ragas.resolve()),'generation_model':run.get('model'),'judge':scored.get('judge'),'expected_retrieval_cases':expected,'summary':{},'excluded_or_missing': [i for i in expected if i not in cases]}
 for metric in METRICS:
  values=[c.get('scores',{}).get(metric) for c in cases.values()]; nums=[v for v in values if isinstance(v,(int,float))]
  missing=[i for i in expected if i not in cases or cases[i].get('scores',{}).get(metric) is None]
  result['summary'][metric]={'scored':len(nums),'expected':len(expected),'mean_available':statistics.mean(nums) if nums else None,'missing_ids':missing}
 result['case_scores']=[{'id':i,'scores':{m:cases.get(i,{}).get('scores',{}).get(m) for m in METRICS},'error':cases.get(i,{}).get('error')} for i in expected]
 path=ROOT/'results'/f"ragas_conversation_summary_v1_{time.strftime('%Y%m%d_%H%M%S')}_{time.time_ns()%1000000:06d}.json"
 path.write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
 print(json.dumps(result['summary'],indent=2));print('Saved:',path)
if __name__=='__main__':main()


## audit_ragas_context_v1.py
Reproducible scoring or audit script.

In [ ]:
%%writefile audit_ragas_context_v1.py
"""Check expected V4 evidence sections in the exact contexts sent to Ragas."""
import argparse,json,time
from pathlib import Path
from check_retrieval_v4 import LABELS
ROOT=Path(__file__).resolve().parent
def main():
 p=argparse.ArgumentParser();p.add_argument('run',type=Path);a=p.parse_args()
 run=json.loads(a.run.read_text(encoding='utf-8')); rows=[]
 for case in run['cases']:
  if case['id'] not in LABELS:continue
  recipe,section=LABELS[case['id']]
  hits=[i for i,m in enumerate(case.get('context_metadata',[]),1) if m.get('recipe')==recipe and m.get('section')==section]
  rows.append({'id':case['id'],'recipe':recipe,'expected_section':section,'rank':hits[0] if hits else None,'chunks':case.get('context_metadata',[])})
 path=ROOT/'results'/f"ragas_context_audit_v1_{time.strftime('%Y%m%d_%H%M%S')}_{time.time_ns()%1000000:06d}.json"
 path.write_text(json.dumps({'source_run':str(a.run.resolve()),'note':'Project-authored section labels; presence of a section does not prove all claims are supported.','cases':rows},indent=2,ensure_ascii=False),encoding='utf-8')
 print('Expected section in provided context:',sum(r['rank'] is not None for r in rows),'/',len(rows));print('Missing:',[r['id'] for r in rows if r['rank'] is None]);print('Saved:',path)
if __name__=='__main__':main()
